✏️ แบบฝึกหัด
ทำทีละข้อในเซลล์ที่เตรียมไว้ให้ แต่ละข้อมีคำใบ้อยู่ท้ายหัวข้อ และมีแนวเฉลยอยู่ในเซลล์สุดท้าย

ข้อ 1 (อุ่นเครื่อง): ประมาณค่าจาก prior sample ล้วน ๆ
ใช้ prior_sample(burglary) อย่างเดียว (ห้ามใช้ rejection_sampling) เพื่อประมาณ P(Alarm = True) แบบไม่มีหลักฐาน แล้วเทียบกับคำตอบแม่นยำจาก enumeration_ask('Alarm', {}, burglary)

คำใบ้: prior_sample คืน dict ของค่าทุกตัวแปร เช่น {'Burglary': False, 'Earthquake': False, 'Alarm': False, ...}

In [ ]:

random.seed(1)

N = 10000
count_alarm = sum(1 for _ in range(N) if prior_sample(burglary)['Alarm'] == T)
count_alarm = 0

p_hat = count_alarm / N
p_exact = enumeration_ask('Alarm', {}, burglary)[True]
print('estimate = %.4f' % p_hat)
print('exact    = %.4f' % p_exact)
print('error    = %.4f' % abs(p_hat - p_exact))

ข้อ 2: เปลี่ยนหลักฐานให้อยู่ "กลางน้ำ"
คำนวณ P(Burglary | Alarm = True) ด้วยทั้ง 3 วิธี ที่ N = 5000 เทียบกับคำตอบแม่นยำ คราวนี้หลักฐานคือ Alarm ซึ่งเป็นพ่อแม่โดยตรงของ JohnCalls และ MaryCalls แทนที่จะเป็นใบล่างสุด

แล้วตอบคำถามในคอมเมนต์ว่า

วิธีไหนแม่นที่สุดในการรันครั้งนี้ และอันดับเปลี่ยนไปจากตารางในขั้นที่ 2 หรือไม่
อัตราการยอมรับของ Alarm=T (จากขั้นที่ 4) ประมาณ 0.2% ซึ่งพอ ๆ กับ JohnCalls=T, MaryCalls=T แล้วทำไม rejection sampling ถึงยังทำได้ไม่ดี ทั้งที่ N = 5000
คำใบ้: ทุกฟังก์ชันมีลายเซ็นเดียวกัน f(X, e, bn, N) จึงวนลูปเรียกได้เลย

In [ ]:

random.seed(2)

e2 = dict(Alarm=T)
exact2 = enumeration_ask('Burglary', e2, burglary)[True]
print('exact P(Burglary=True | Alarm=T) = %.4f\n' % exact2)

for name, fn in methods:
    p, _t = estimate2(fn, 5000)
    print('%-12s %.4f  (error %.4f)' % (name, p, abs(p - exact2)))

ข้อ 3: ต้องใช้ตัวอย่างกี่ตัวถึงจะ "แม่นพอ"
หาค่า N ที่น้อยที่สุดจากลิสต์ [100, 200, 500, 1000, 2000, 5000, 10000] ที่ทำให้ likelihood_weighting มีค่าคลาดเคลื่อนสัมบูรณ์ เฉลี่ยจาก 20 รอบ น้อยกว่า 0.01 สำหรับคำถาม P(Burglary | JohnCalls=T, MaryCalls=T)

จากนั้นทำแบบเดียวกันกับ gibbs_ask แล้วเปรียบเทียบว่าอันไหนต้องใช้ตัวอย่างน้อยกว่ากัน ถ้าวิธีไหนไม่ผ่านเกณฑ์เลยแม้ที่ N = 10000 ให้รายงานว่า None และอธิบายว่าเพราะอะไร

คำใบ้: โครงลูปเหมือนเซลล์วาดกราฟด้านบน ใช้ return N ทันทีเมื่อเจอ N ตัวแรกที่ผ่านเกณฑ์ และอย่าลืมครอบ try/except ZeroDivisionError ถ้าจะเอาไปใช้กับ rejection_sampling ด้วย

In [ ]:
random.seed(3)

exact3 = enumeration_ask('Burglary', {'JohnCalls': T, 'MaryCalls': T}, burglary)[True]

def samples_needed(fn, threshold=0.01, repeats=20):
    # คืน N ตัวแรกที่ทำให้ค่าคลาดเคลื่อนเฉลี่ยต่ำกว่า threshold, คืน None ถ้าไม่มี
    for N in [100, 200, 500, 1000, 2000, 5000, 10000]:
        errors: list[float] = []

        for _ in range(repeats):
            p = fn('Burglary', {'JohnCalls': T, 'MaryCalls': T}, burglary, N)[True]
            errors.append(abs(p - exact3))

        mean_err = statistics.mean(errors)
        print('  N=%-6d mean error = %.4f' % (N, mean_err))

        if mean_err < threshold:
            return N

    return None


print('likelihood_weighting:')
print('  -> ต้องใช้ N =', samples_needed(likelihood_weighting))
print('gibbs_ask:')
print('  -> ต้องใช้ N =', samples_needed(gibbs_ask))

ข้อ 4 (ท้าทาย): เขียน likelihood weighting ด้วยตัวเอง
เขียนฟังก์ชัน my_likelihood_weighting(X, e, bn, N) ให้ได้ผลเหมือน likelihood_weighting โดยใช้ weighted_sample(bn, e) ที่มีอยู่แล้ว ซึ่งคืนคู่ (event, weight)

ขั้นตอน (AIMA รูป 14.15):

สร้าง dict W เก็บน้ำหนักสะสมของแต่ละค่าที่ X เป็นไปได้ (bn.variable_values(X))
วน N รอบ: เรียก weighted_sample ได้ (sample, w) แล้วบวก w เข้าไปที่ W[sample[X]]
คืน ProbDist(X, W) ซึ่งจะ normalize ให้เองอัตโนมัติ
คำใบ้: ถ้าติด ให้ดูโค้ดต้นฉบับด้วย psource(likelihood_weighting) หลังจาก ลองเขียนเองแล้ว

In [ ]:

def my_likelihood_weighting(X, e, bn, N=1000):
    W = {x: 0.0 for x in bn.variable_values(X)}
    for _ in range(N):
          sample, weight = weighted_sample(bn, e)
          W[sample[X]] += weight
        pass
    return ProbDist(X, W)


# ทดสอบแบบเข้มงวด: ถ้าเขียนถูก จะเรียก weighted_sample ในลำดับเดียวกับของจริงเป๊ะ ๆ
# ตั้ง seed เดียวกันแล้วผลลัพธ์ต้องตรงกันทุกทศนิยม
random.seed(4)
result = my_likelihood_weighting('Burglary', evidence, burglary, 20000)
random.seed(4)
reference = likelihood_weighting('Burglary', evidence, burglary, 20000)

print('my_likelihood_weighting =', result.show_approx())
print('likelihood_weighting    =', reference.show_approx())
print('exact                   =', exact.show_approx())
assert abs(result[True] - reference[True]) < 1e-12, 'ยังไม่ตรงกับของจริง ลองตรวจการสะสมน้ำหนักอีกครั้ง'
print('ผ่าน')